# VISTA Demo: POPE Evaluation with LLaVA 1.5 7B

This notebook runs the **VISTA** pipeline on the **POPE** dataset using **LLaVA 1.5 7B** for a **few examples** (quick demo).

**Prerequisites:**
- **LLaVA 1.5 7B**: Set `LLAVA_MODEL_PATH` in the config cell to your checkpoint (e.g. `../download_models/llava-v1.5-7b` or absolute path). The repo default is `../download_models/llava-v1.5-7b` relative to the VISTA folder.
- **COCO**: Set `COCO_DATA_PATH` to your MSCOCO 2014 val images folder (containing `COCO_val2014_*.jpg`).

**Options:**
- `NUM_EXAMPLES`: number of POPE samples (e.g. 5–10 for a quick run).
- `USE_VSV`: Visual Steering Vectors (2 extra forwards per example).
- `USE_LOGITS_AUG`: Self-Logits Augmentation (SLA).
- `POPE_TYPE`: `"random"`, `"popular"`, or `"adversarial"`.

In [ ]:
# Configuration: run this cell first and set paths for your environment
import os
import sys

# Ensure VISTA root is on the path. If you open the notebook from VISTA/, cwd is usually VISTA/.
VISTA_ROOT = os.getcwd()
if os.path.basename(VISTA_ROOT) != 'VISTA':
    # If cwd is project root, use VISTA subfolder
    VISTA_ROOT = os.path.join(os.getcwd(), 'VISTA')
    if not os.path.isdir(VISTA_ROOT):
        VISTA_ROOT = os.getcwd()
sys.path.insert(0, VISTA_ROOT)
os.chdir(VISTA_ROOT)

# --- LLaVA 1.5 7B: set path to your checkpoint ---
LLAVA_MODEL_PATH = os.path.expanduser("../download_models/llava-v1.5-7b")  # or absolute path, e.g. "/data/llava-v1.5-7b"
MODEL_NAME = "llava-1.5"  # LLaVA 1.5 7B (do not change for this demo)

# --- Data ---
COCO_DATA_PATH = os.path.expanduser("../download_datasets/COCO_2014/val2014")  # MSCOCO 2014 val images
POPE_TYPE = "random"     # one of: random, popular, adversarial

# --- Few examples for quick demo ---
NUM_EXAMPLES = 5
MAX_NEW_TOKENS = 32
SEED = 1994

# --- VISTA options ---
USE_VSV = True
USE_LOGITS_AUG = True
VSV_LAMBDA = 0.01
LOGITS_LAYERS = "25,30"
LOGITS_ALPHA = 0.3

print("VISTA_ROOT:", VISTA_ROOT)
print("LLaVA 7B path:", LLAVA_MODEL_PATH)
print("COCO_DATA_PATH:", COCO_DATA_PATH)
print("NUM_EXAMPLES:", NUM_EXAMPLES)

In [ ]:
# Imports (must run after path setup)
import json
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

import myutils
from anchor import POPE_PATH
from eval_data_loader import POPEDataSet
from llava.utils import disable_torch_init
from model_loader import ModelLoader
from steering_vector import obtain_vsv, add_logits_flag, remove_logits_flag
from llm_layers import add_vsv_layers, remove_vsv_layers

In [ ]:
# Seed and disable torch init for reproducibility / faster load
myutils.seed_everything(SEED)
disable_torch_init()

# Use notebook's LLaVA path for LLaVA 7B (override model_loader default)
import model_loader as _ml
_orig_load_model = _ml.ModelLoader.load_model
def _patched_load_model(self):
    if self.model_name == "llava-1.5":
        path = os.path.expanduser(LLAVA_MODEL_PATH)
        if not os.path.exists(path):
            raise FileNotFoundError(f"LLaVA 7B checkpoint not found at {path}. Set LLAVA_MODEL_PATH in the config cell.")
        self.tokenizer, self.vlm_model, self.image_processor, self.llm_model = _ml.load_llava_model(path)
        return
    _orig_load_model(self)
_ml.ModelLoader.load_model = _patched_load_model

# Load LLaVA 1.5 7B (this may take a minute and requires GPU)
print(f"Loading {MODEL_NAME} (LLaVA 7B) from {LLAVA_MODEL_PATH}...")
model_loader = ModelLoader(MODEL_NAME)
print("Model loaded.")

In [ ]:
# Load POPE dataset and take a small subset
pope_path = POPE_PATH[POPE_TYPE]
pope_dataset = POPEDataSet(
    pope_path=pope_path,
    data_path=COCO_DATA_PATH,
    trans=model_loader.image_processor,
)
if NUM_EXAMPLES > 0 and NUM_EXAMPLES < len(pope_dataset):
    indices = np.random.choice(len(pope_dataset), NUM_EXAMPLES, replace=False)
    pope_dataset = Subset(pope_dataset, indices)

pope_loader = DataLoader(
    pope_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)
template = myutils.prepare_template(type('Args', (), {'model': MODEL_NAME})())

print(f"POPE type: {POPE_TYPE}, number of examples: {len(pope_dataset)}")

In [ ]:
# Build a minimal args object for steering/logits options
class Args:
    pass
args = Args()
args.vsv = USE_VSV
args.vsv_lambda = VSV_LAMBDA
args.layers = None
args.logits_aug = USE_LOGITS_AUG
args.logits_layers = LOGITS_LAYERS
args.logits_alpha = LOGITS_ALPHA
args.model = MODEL_NAME
args.do_sample = False
args.top_p = 0.9
args.top_k = 0
args.max_new_tokens = MAX_NEW_TOKENS
args.num_beams = 1
args.no_repeat_ngram_size = None
args.temperature = 1.0
args.repetition_penalty = 1.0

In [ ]:
# Run inference with LLaVA 7B for the configured few examples
from tqdm import tqdm
results = []

for batch_idx, data in tqdm(enumerate(pope_loader), total=len(pope_loader), desc="POPE (LLaVA 7B)"):
    image = data["image"]
    query = data["query"]
    label = data["label"].tolist()

    with torch.inference_mode():
        with myutils.maybe_autocast(MODEL_NAME, model_loader.vlm_model.device):
            questions, kwargs = model_loader.prepare_inputs_for_model(template, query, image)

            if args.vsv:
                neg_kwargs = model_loader.prepare_neg_prompt(args, questions, template=template)
                pos_kwargs = model_loader.prepare_pos_prompt(args, kwargs)
                visual_vector, _ = obtain_vsv(args, model_loader.llm_model, [[neg_kwargs, pos_kwargs]])
                add_vsv_layers(model_loader.llm_model, torch.stack([visual_vector], dim=1).cuda(), [args.vsv_lambda])

            add_logits_flag(model_loader.llm_model, args)

            gen_kwargs = dict(kwargs)
            if args.do_sample:
                gen_kwargs["top_p"] = args.top_p
                gen_kwargs["top_k"] = args.top_k

            outputs = model_loader.llm_model.generate(
                do_sample=args.do_sample,
                max_new_tokens=args.max_new_tokens,
                use_cache=True,
                num_beams=args.num_beams,
                output_attentions=False,
                output_hidden_states=True if args.logits_aug else False,
                no_repeat_ngram_size=args.no_repeat_ngram_size,
                temperature=args.temperature,
                repetition_penalty=args.repetition_penalty,
                return_dict=True,
                **gen_kwargs,
            )

            remove_logits_flag(model_loader.llm_model)
            if args.vsv:
                remove_vsv_layers(model_loader.llm_model)

    output_text = model_loader.decode(outputs)

    for i in range(len(output_text)):
        results.append({
            "query": query[i],
            "label": "yes" if label[i] == 1 else "no",
            "pred": output_text[i],
            "question": questions[i],
        })

print(f"Completed {len(results)} examples.")

In [ ]:
# Display results (normalize pred to yes/no for comparison)
def normalize_ans(txt):
    t = txt.strip().lower()
    if t.startswith("yes") or t == "y":
        return "yes"
    if t.startswith("no") or t == "n":
        return "no"
    return txt

correct = 0
for r in results:
    pred_norm = normalize_ans(r["pred"])
    ok = pred_norm == r["label"]
    if ok:
        correct += 1
    print(f"Q: {r['query']}")
    print(f"   Label: {r['label']}  Pred: {r['pred']}  -> {pred_norm}  {"✓" if ok else "✗"}")
    print()

if results:
    print(f"Accuracy: {correct}/{len(results)} = {100.0 * correct / len(results):.1f}%")